In [2]:
import re
import pandas as pd
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# Mapa de meses para parsear fechas en español
month_map = {
    'enero':'01','febrero':'02','marzo':'03','abril':'04',
    'mayo':'05','junio':'06','julio':'07','agosto':'08',
    'septiembre':'09','octubre':'10','noviembre':'11','diciembre':'12'
}

def parse_date_spanish(date_str: str) -> str:
    """Convierte fechas como "20 de Julio de 1995" → "20/07/1995"."""
    s = date_str.strip().lower()
    m = re.match(r"(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})", s)
    if m:
        day, mon, year = m.groups()
        return f"{int(day):02d}/{month_map.get(mon,'00')}/{year}"
    m = re.match(r"(\w+)\s+(\d{1,2})\s+de\s+(\d{4})", s)
    if m:
        mon, day, year = m.groups()
        return f"{int(day):02d}/{month_map.get(mon,'00')}/{year}"
    return date_str

def extract_text_txt(path: str) -> str:
    """Lee todo el contenido de un .txt y lo devuelve."""
    with open(path, encoding='utf-8') as f:
        return f.read()

# 1) Lee y limpia líneas “LEYES <número>”
raw = extract_text_txt('legislatura_1995_1996.txt')
lines = []
for ln in raw.splitlines():
    if not re.match(r'^\s*LEYES\s+\d+\s*$', ln, flags=re.IGNORECASE):
        lines.append(ln)
clean = "\n".join(lines)

# 2) Divide en bloques que comienzan con “PROYECTO DE”
bloques = re.split(r'(?=PROYECTO\s+DE)', clean, flags=re.IGNORECASE)

rows = []
for bloque in bloques:
    text = bloque.strip()
    if not text:
        continue

    # Cámara No. y año
    m = re.search(
        r"PROYECTO\s+DE(?:\s+ACTO)?\s+(?:LEGISLATIVO|LEY)\s+(\d+)\s+DE\s+(\d{4})\s+CAMARA",
        text, flags=re.IGNORECASE
    )
    cam_no, cam_year = m.groups() if m else ("","")

    # ACUM (si existe)
    m = re.search(r"CAMARA-ACUMULADO:\s*([\d/]+\s*-\s*[\d/]+)", text, flags=re.IGNORECASE)
    acum = m.group(1).replace(" ", "") if m else ""

    # Senado No. y año
    m = re.search(r"(\d{1,3})/(\d{2,4})\s+SENADO", text, flags=re.IGNORECASE)
    sen_no, sen_year = m.groups() if m else ("","")

    # Fecha Rad. Cámara (PRESENTACION)
    m = re.search(
        r"PRESENTACION\s+([A-Za-záéíóúÁÉÍÓÚñÑ]+\s+\d{1,2}\s+DE\s+\d{4})",
        text, flags=re.IGNORECASE
    )
    fcam = parse_date_spanish(m.group(1)) if m else ""

    # Estado Actual
    m = re.search(r"ESTADO ACTUAL\s+([^\n]+)", text, flags=re.IGNORECASE)
    est = m.group(1).strip() if m else ""

    # TITULO (multilínea hasta ORIGEN)
    m = re.search(r'TITULO\s*"([\s\S]*?)"\s*ORIGEN', text, flags=re.IGNORECASE)
    titulo = " ".join(m.group(1).split()) if m else ""

    # ORIGEN
    m = re.search(r'ORIGEN\s+([^\n]+)', text, flags=re.IGNORECASE)
    origen = m.group(1).strip() if m else ""

    # AUTOR
    m = re.search(r'AUTOR(?:ES)?:\s*([^\n]+)', text, flags=re.IGNORECASE)
    autor = m.group(1).strip() if m else ""

    # PONENTE(s) (multilínea hasta COMISION)
    m = re.search(r'PONENTES?:\s*([\s\S]*?)(?=\nCOMISION)', text, flags=re.IGNORECASE)
    ponentes = " ".join(m.group(1).split()) if m else ""

    # COMISION
    m = re.search(r'COMISION\s+([^\n]+)', text, flags=re.IGNORECASE)
    comision = m.group(1).strip() if m else ""

    # PUBLICACION PROYECTO GACETA
    m = re.search(r'PUBLICACION\s+PROYECTO\s+GACETA\s*No\.?(\d+)/(\d+)', text, flags=re.IGNORECASE)
    pub_proj_num, pub_proj_year = m.groups() if m else ("","")

    # PONENCIA PRIMER DEBATE
    m = re.search(r'PONENCIA\s+PRIMER\s+DEBATE.*?No\.?([^\n]+)', text, flags=re.IGNORECASE|re.DOTALL)
    primer = m.group(1) if m else ""
    p1_cam_num = p1_cam_year = p1_sen_num = p1_sen_year = ""
    for num, year, let in re.findall(r'(\d{1,3})/(\d{2,4})(?:\s*([cCsS]))', primer):
        if let.upper()=='C':
            p1_cam_num, p1_cam_year = num, year
        else:
            p1_sen_num, p1_sen_year = num, year

    # PONENCIA SEGUNDO DEBATE
    m = re.search(r'PONENCIA\s+SEGUNDO\s+DEBATE.*?No\.?([^\n]+)', text, flags=re.IGNORECASE|re.DOTALL)
    segundo = m.group(1) if m else ""
    p2_cam_num = p2_cam_year = p2_sen_num = p2_sen_year = ""
    for num, year, let in re.findall(r'(\d{1,3})/(\d{2,4})(?:\s*([cCsS]))', segundo):
        if let.upper()=='C':
            p2_cam_num, p2_cam_year = num, year
        else:
            p2_sen_num, p2_sen_year = num, year

    rows.append({
        'Camara No':         cam_no,
        'Camara año':        cam_year,
        'ACUM':              acum,
        'Senado No':         sen_no,
        'Senado año':        sen_year,
        'Fecha Rad. Cámara': fcam,
        'Estado Actual':     est,
        'Titulo':            titulo,
        'Origen':            origen,
        'Autor':             autor,
        'Ponentes':          ponentes,
        'Comision':          comision,
        'Pub_Proj_Num':      pub_proj_num,
        'Pub_Proj_Year':     pub_proj_year,
        'P1_Cam_Num':        p1_cam_num,
        'P1_Cam_Year':       p1_cam_year,
        'P1_Sen_Num':        p1_sen_num,
        'P1_Sen_Year':       p1_sen_year,
        'P2_Cam_Num':        p2_cam_num,
        'P2_Cam_Year':       p2_cam_year,
        'P2_Sen_Num':        p2_sen_num,
        'P2_Sen_Year':       p2_sen_year,
    })

# 3) Armado del DataFrame y guardado
cols = [
    'Camara No','Camara año','ACUM','Senado No','Senado año',
    'Fecha Rad. Cámara','Estado Actual','Titulo','Origen',
    'Autor','Ponentes','Comision',
    'Pub_Proj_Num','Pub_Proj_Year',
    'P1_Cam_Num','P1_Cam_Year','P1_Sen_Num','P1_Sen_Year',
    'P2_Cam_Num','P2_Cam_Year','P2_Sen_Num','P2_Sen_Year'
]
df = pd.DataFrame(rows, columns=cols)

# Limpieza de caracteres inválidos
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].apply(lambda s: ILLEGAL_CHARACTERS_RE.sub('', s) if isinstance(s, str) else s)

# Guardar a Excel
df.to_excel('legislatura_dataClean.xlsx', index=False)
print("✅ fichas_formato2_completo.xlsx generado.")


✅ fichas_formato2_completo.xlsx generado.


In [ ]:
#1995_1996 Legislatura

import re
import pandas as pd
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# — Mapa para convertir meses en números —
month_map = {
    'enero':'01','febrero':'02','marzo':'03','abril':'04',
    'mayo':'05','junio':'06','julio':'07','agosto':'08',
    'septiembre':'09','octubre':'10','noviembre':'11','diciembre':'12'
}

def parse_date_spanish(date_str: str) -> str:
    """
    Convierte "20 de Julio de 1995" o "JULIO 20 DE 1995" → "20/07/1995".
    """
    s = date_str.strip().lower()
    m = re.match(r"(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})", s)
    if m:
        day, mon, year = m.groups()
        return f"{int(day):02d}/{month_map.get(mon,'00')}/{year}"
    m = re.match(r"(\w+)\s+(\d{1,2})\s+de\s+(\d{4})", s)
    if m:
        mon, day, year = m.groups()
        return f"{int(day):02d}/{month_map.get(mon,'00')}/{year}"
    return date_str

def extract_text_txt(path: str) -> str:
    """Lee todo el .txt y devuelve una sola cadena."""
    with open(path, encoding='utf-8') as f:
        return f.read()

def extract_section(text: str, start: str, end: str) -> str:
    """
    Captura multilínea entre `start` y `end` (sin incluir `end`),
    colapsa saltos de línea y espacios en blanco.
    Si end es vacío, captura hasta el final.
    """
    if end:
        pat = rf"{re.escape(start)}\s*([\s\S]*?)(?={re.escape(end)})"
    else:
        pat = rf"{re.escape(start)}\s*([\s\S]*)"
    m = re.search(pat, text, flags=re.IGNORECASE)
    return " ".join(m.group(1).split()) if m else ""

# 1) Leer y limpiar líneas "LEYES <número>"
raw = extract_text_txt('legislatura_1995_1996.txt')
lines = [
    ln for ln in raw.splitlines()
    if not re.match(r'^\s*LEYES\s+\d+\s*$', ln, flags=re.IGNORECASE)
]
clean = "\n".join(lines)

# 2) Partir cada ficha por "PROYECTO DE"
bloques = re.split(r"(?=PROYECTO\s+DE)", clean, flags=re.IGNORECASE)

rows = []
for bloque in bloques:
    txt = bloque.strip()
    if not txt:
        continue

    # — Encabezado: PROYECTO DE … CAMARA (y opcional “-ACUMULADO”) y SENADO —
    header = txt.splitlines()[0]
    m = re.search(
        r"PROYECTO\s+DE(?:\s+ACTO)?\s+(?:LEGISLATIVO|LEY)\s+(\d+)\s+DE\s+(\d{4})\s+CAMARA",
        header, flags=re.IGNORECASE
    )
    cam_no, cam_year = (m.group(1), m.group(2)) if m else ("","")
    m2 = re.search(r"CAMARA-ACUMULADO:\s*([\d/]+\s*-\s*[\d/]+)", header, flags=re.IGNORECASE)
    acum = m2.group(1).replace(" ", "") if m2 else ""
    m3 = re.search(r"(\d{1,3})/(\d{2,4})\s+SENADO", header, flags=re.IGNORECASE)
    sen_no, sen_year = (m3.group(1), m3.group(2)) if m3 else ("","")

    # — Campos multilínea —
    tit  = extract_section(txt, "TITULO",   "ORIGEN").strip('"')
    ori  = extract_section(txt, "ORIGEN",   "AUTOR")
    aut  = extract_section(txt, "AUTOR",    "PRESENTACION")
    pon  = extract_section(txt, "PONENTE",  "COMISION")
    com  = extract_section(txt, "COMISION", "PUBLICACIONES")

    # — Fecha de Radicación Cámara —
    m4 = re.search(r"PRESENTACION\s+([\s\S]*?)(?=PONENTE)", txt, flags=re.IGNORECASE)
    fcam = parse_date_spanish(m4.group(1)) if m4 else ""

    # — Estado Actual —
    est = extract_section(txt, "ESTADO ACTUAL", "")

    # — Publicaciones: bloque entre PUBLICACIONES y ESTADO ACTUAL —
    pub = extract_section(txt, "PUBLICACIONES", "ESTADO ACTUAL")

    # Publicacion PROYECTO GACETA
    m5 = re.search(r"PROYECTO.*?No\.\s*(\d{1,3})/(\d{2,4})", pub, flags=re.IGNORECASE)
    pproy_no, pproy_yr = (m5.group(1), m5.group(2)) if m5 else ("","")

    # Ponencia Primer Debate
    blk1 = extract_section(pub, "PONENCIA PRIMER DEBATE", "PONENCIA SEGUNDO")
    codes1_cam = re.findall(r"(\d{1,3})/(\d{2,4})(?=\s*[cC])", blk1, flags=re.IGNORECASE)
    codes1_sen = re.findall(r"(\d{1,3})/(\d{2,4})(?=\s*[sS])", blk1, flags=re.IGNORECASE)
    if codes1_cam:
        pon1_cam_no, pon1_cam_yr = codes1_cam[0]
    else:
        pon1_cam_no = pon1_cam_yr = ""
    if codes1_sen:
        pon1_sen_no, pon1_sen_yr = codes1_sen[0]
    else:
        pon1_sen_no = pon1_sen_yr = ""

    # Ponencia Segundo Debate
    blk2 = extract_section(pub, "PONENCIA SEGUNDO DEBATE", "")
    codes2_cam = re.findall(r"(\d{1,3})/(\d{2,4})(?=\s*[cC])", blk2, flags=re.IGNORECASE)
    codes2_sen = re.findall(r"(\d{1,3})/(\d{2,4})(?=\s*[sS])", blk2, flags=re.IGNORECASE)
    if codes2_cam:
        pon2_cam_no, pon2_cam_yr = codes2_cam[0]
    else:
        pon2_cam_no = pon2_cam_yr = ""
    if codes2_sen:
        pon2_sen_no, pon2_sen_yr = codes2_sen[0]
    else:
        pon2_sen_no = pon2_sen_yr = ""

    rows.append({
        'Camara No':        cam_no,
        'Camara año':       cam_year,
        'ACUM':             acum,
        'Senado No':        sen_no,
        'Senado año':       sen_year,
        'Fecha Rad. Cámara':fcam,
        'Estado Actual':    est,
        'Titulo':           tit,
        'Origen':           ori,
        'Autor':            aut,
        'Ponente':          pon,
        'Comision':         com,
        'Proy Gaceta No':   pproy_no,
        'Proy Gaceta Año':  pproy_yr,
        'Pon1 Cam No':      pon1_cam_no,
        'Pon1 Cam Año':     pon1_cam_yr,
        'Pon1 Sen No':      pon1_sen_no,
        'Pon1 Sen Año':     pon1_sen_yr,
        'Pon2 Cam No':      pon2_cam_no,
        'Pon2 Cam Año':     pon2_cam_yr,
        'Pon2 Sen No':      pon2_sen_no,
        'Pon2 Sen Año':     pon2_sen_yr,
    })

# 3) Crear DataFrame y exportar
df = pd.DataFrame(rows)
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].apply(
        lambda s: ILLEGAL_CHARACTERS_RE.sub('', s) if isinstance(s, str) else s
    )
df.to_excel('fichas_formato2_mejoradoV2.xlsx', index=False)


In [12]:
import re
import pandas as pd
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# — Mapa para convertir meses a número —
month_map = {
    'enero':'01','febrero':'02','marzo':'03','abril':'04',
    'mayo':'05','junio':'06','julio':'07','agosto':'08',
    'septiembre':'09','octubre':'10','noviembre':'11','diciembre':'12'
}

def parse_date_spanish(s: str) -> str:
    """
    Normaliza fechas como:
      - "20 de Julio de 1995"
      - "JUNIO 18 DE 1.993"
    → "20/07/1995", "18/06/1993"
    Elimina puntos en el año.
    """
    s = s.replace('.', '').strip().lower()
    # dd de mes de yyyy
    m = re.match(r"(\d{1,2})\s+de\s+(\w+)\s+de\s+(\d{4})", s)
    if m:
        d, mon, y = m.groups()
        return f"{int(d):02d}/{month_map.get(mon,'00')}/{y}"
    # mes dd de yyyy
    m = re.match(r"(\w+)\s+(\d{1,2})\s+de\s+(\d{4})", s)
    if m:
        mon, d, y = m.groups()
        return f"{int(d):02d}/{month_map.get(mon,'00')}/{y}"
    return s

def extract_txt(path: str) -> str:
    """Lee todo el contenido de un .txt y retorna un string."""
    with open(path, encoding='utf-8') as f:
        return f.read()

def extract_section(text: str, start: str, end: str) -> str:
    """
    Captura multilínea entre start y end (sin incluir end),
    colapsa saltos de línea y espacios.
    Si end es vacío, captura hasta el final.
    """
    if end:
        pat = rf"{re.escape(start)}\s*([\s\S]*?)(?={re.escape(end)})"
    else:
        pat = rf"{re.escape(start)}\s*([\s\S]*)"
    m = re.search(pat, text, flags=re.IGNORECASE)
    return " ".join(m.group(1).split()) if m else ""

# 1) Leer y limpiar el texto
raw = extract_txt('legislatura_1993_1994_1995.txt')
lines = []
for ln in raw.splitlines():
    # ignorar marcadores de error, líneas "LEYES <n>" y líneas vacías
    if re.match(r'^\s*¡Error! Marcador no definido\.', ln): continue
    if re.match(r'^\s*LEYES\s+\d+\s*$', ln, flags=re.IGNORECASE): continue
    if not ln.strip(): continue
    lines.append(ln)
clean = "\n".join(lines)

# 2) Dividir en bloques por "PROYECTO DE"
bloques = re.split(r"(?=PROYECTO\s+DE)", clean, flags=re.IGNORECASE)

rows = []
for bloque in bloques:
    txt = bloque.strip()
    if not txt: continue

    # Tipo de Ley
    m = re.search(r"PROYECTO\s+DE\s+(ACTO\s+LEGISLATIVO|LEY)", txt, flags=re.IGNORECASE)
    tipo = m.group(1).title() if m else ""

    # Encabezado (línea 1)
    header = txt.splitlines()[0]

    # Cámara No. y año
    m = re.search(r"No\.?\s*(\d{1,3})/(\d{2,4})\s+CAMARA", header, flags=re.IGNORECASE)
    cam_no, cam_year = (m.group(1).lstrip('0'), m.group(2)) if m else ("","")

    # ACUM C en paréntesis o tras "ACUM."
    acum_c = ""
    m_ac1 = re.search(r"\(ACUM[^\)]*\)", txt, flags=re.IGNORECASE)
    if m_ac1:
        codes = re.findall(r"(\d{1,3}/\d{2,4})", m_ac1.group(0))
        acum_c = ", ".join(codes)
    else:
        m_ac2 = re.search(r"ACUM\.?\s+([\d/, -]+)", txt, flags=re.IGNORECASE)
        if m_ac2:
            codes = re.findall(r"(\d{1,3}/\d{2,4})", m_ac2.group(1))
            acum_c = ", ".join(codes)

    # Senado No. y año
    m = re.search(r"(\d{1,3})/(\d{2,4})\s+SENADO", txt, flags=re.IGNORECASE)
    sen_no, sen_year = (m.group(1).lstrip('0'), m.group(2)) if m else ("N/A","N/A")

    # Título
    titulo = extract_section(txt, "TITULO", "ORIGEN").strip('"')

    # Origen
    m = re.search(r"ORIGEN[:]? ?([^\n]+)", txt, flags=re.IGNORECASE)
    origen = m.group(1).strip() if m else ""

    # Autor
    autor = extract_section(txt, "AUTOR[:]?","PRESENTACION")

    # Fecha Radicación (PRESENTACION)
    m = re.search(r"PRESENTACION[:]? ?([\w\s\d\.\-]+?)(?=\s+COMISION|\s+PONENTE|\s+PUBLICACION)",
                  txt, flags=re.IGNORECASE)
    fecha_rad = parse_date_spanish(m.group(1)) if m else ""

    # Comisión
    m = re.search(r"COMISION[:]? ?([^\n]+)", txt, flags=re.IGNORECASE)
    comision = m.group(1).strip() if m else ""

    # Ponentes
    ponentes = extract_section(txt, "PONENTE[S]?:?", "PUBLICACION|ESTADO ACTUAL")
    if not ponentes:
        ponentes = "N/A"

    # Publicaciones Gaceta
    pub = extract_section(txt, "PUBLICACION", "ESTADO ACTUAL")
    # extraer todos los códigos x/y
    gaceta_codes = re.findall(r"(\d{1,3})/(\d{2,4})", pub)
    publica_gaceta = ", ".join(f"{num}/{yr}" for num,yr in gaceta_codes)

    # Estado Actual
    estado = extract_section(txt, "ESTADO ACTUAL[:]?","")

    rows.append({
        'Tipo de Ley':       tipo,
        'Camara No':         cam_no,
        'Camara Anio':       cam_year,
        'ACUM C':            acum_c or "N/A",
        'Senado No':         sen_no,
        'Senado Anio':       sen_year,
        'Titulo':            titulo,
        'Origen':            origen,
        'Autor':             autor,
        'Fecha Rad.':        fecha_rad,
        'Comision':          comision,
        'Ponentes':          ponentes,
        'Publica_Gaceta_No': publica_gaceta,
        'Estado Actual':     estado
    })

# 3) DataFrame y guardado
df = pd.DataFrame(rows)
for col in df.select_dtypes(include='object'):
    df[col] = df[col].apply(
        lambda s: ILLEGAL_CHARACTERS_RE.sub('', s) if isinstance(s,str) else s
    )
df.to_excel('legislatura_1993_1994_1995.xlsx', index=False)


In [20]:
import re
import pandas as pd
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# — Meses a número —
month_map = {
    'enero':'01','febrero':'02','marzo':'03','abril':'04',
    'mayo':'05','junio':'06','julio':'07','agosto':'08',
    'septiembre':'09','octubre':'10','noviembre':'11','diciembre':'12'
}

def parse_date_1993(s: str) -> str:
    """
    Convierte:
      - "JULIO 20/93"      → "20/07/1993"
      - "NOVIEMBRE 04 DE 1.993" → "04/11/1993"
    Asume siglo 1900 para /93.
    """
    txt = s.strip().replace('.', '').upper()
    # formato "MES DD/YY"
    m = re.match(r"([A-ZÁÉÍÓÚÑ]+)\s+(\d{1,2})/(\d{2})", txt)
    if m:
        mon, day, yy = m.groups()
        yyyy = '19' + yy
        return f"{int(day):02d}/{month_map.get(mon.lower(),'00')}/{yyyy}"
    # formato "MES DD DE YYYY"
    m = re.match(r"([A-ZÁÉÍÓÚÑ]+)\s+(\d{1,2})\s+DE\s+(\d{4})", txt)
    if m:
        mon, day, yyyy = m.groups()
        return f"{int(day):02d}/{month_map.get(mon.lower(),'00')}/{yyyy}"
    return s.strip()

def extract_txt(path: str) -> str:
    with open(path, encoding='utf-8') as f:
        return f.read()

def extract_section(text: str, start: str, end: str="") -> str:
    """
    Captura multilínea entre start y end (sin incluir end),
    normalizando espacios. Si end="" captura hasta final.
    """
    if end:
        pat = rf"{re.escape(start)}\s*([\s\S]*?)(?={end})"
    else:
        pat = rf"{re.escape(start)}\s*([\s\S]*)"
    m = re.search(pat, text, flags=re.IGNORECASE)
    return " ".join(m.group(1).split()) if m else ""

# 1) Leer y limpiar
raw = extract_txt('legislatura_1993.txt')
lines = []
for ln in raw.splitlines():
    ln = ln.strip()
    if not ln: continue
    if ln.startswith("¡Error! Marcador no definido."): continue
    if re.match(r'^LEYES\s+\d+', ln, re.IGNORECASE): continue
    lines.append(ln)
clean = "\n".join(lines)

# 2) Separar por proyecto
bloques = re.split(r"(?=PROYECTO\s+DE\s+LEY)", clean, flags=re.IGNORECASE)

rows = []
for blk in bloques:
    txt = blk.strip()
    if not txt: continue

    # Proyecto No. y Año (independiente de cámara/senado)
    m = re.search(r"No\.?\s*(\d{1,3})/([\d\.]+)", txt, flags=re.IGNORECASE)
    if m:
        proj_no = m.group(1).lstrip('0')
        yy_raw = m.group(2).replace('.', '')
        proj_year = '19' + yy_raw if len(yy_raw)==2 else yy_raw
    else:
        proj_no = proj_year = ""

    # Tipo de proyecto
    tipo = "Ley"
    if re.search(r"ACTO\s+LEGISLATIVO", txt, flags=re.IGNORECASE):
        tipo = "Acto Legislativo"

    # Cámara No. y año (si aparece)
    m = re.search(r"No\.?\s*\d{1,3}/\d{2,4}\s+CAMARA", txt, flags=re.IGNORECASE)
    if m:
        cam_match = re.search(r"(\d{1,3})/(\d{2,4})\s+CAMARA", m.group(0), flags=re.IGNORECASE)
        cam_no, cam_yy = cam_match.groups()
        cam_no = cam_no.lstrip('0')
        cam_year = '19'+cam_yy if len(cam_yy)==2 else cam_yy
    else:
        cam_no = cam_year = "N/A"

    # Senado No. y año
    m = re.search(r"^(\d{1,3})/(\d{2,4})\s+SENADO", txt, flags=re.IGNORECASE|re.MULTILINE)
    if m:
        sen_no, sen_yy = m.groups()
        sen_no = sen_no.lstrip('0')
        sen_year = '19'+sen_yy if len(sen_yy)==2 else sen_yy
    else:
        sen_no = sen_year = "N/A"

    # Título
    titulo = extract_section(txt, "TITULO", "AUTOR").strip('"')

    # Autor (entre AUTOR y PRESENTAD)
    autor = extract_section(txt, "AUTOR", "PRESENT").strip()
    if not autor:
        # si no hay AUTOR: algunos usan "AUTOR:"
        autor = extract_section(txt, "AUTOR:", "PRESENT").strip()

    # Fecha Radicación (presentado/presentación)
    pres_block = extract_section(txt, "PRESENTACION", "COMISION")

    # 2) Extrae todos los posibles formatos de fecha:
    #    - "16 de diciembre de 1993"
    #    - "NOVIEMBRE 30/93"
    raw_dates = re.findall(
        r"\d{1,2}\s+de\s+\w+\s+de\s+\d{4}"   # dd de mes de yyyy
        r"|[A-ZÁÉÍÓÚÑ]+\s+\d{1,2}/\d{2}",      # MES dd/yy
        pres_block,
        flags=re.IGNORECASE
    )

    # 3) Normaliza cada uno a DD/MM/AAAA
    fechas = [parse_date_1993(d) for d in raw_dates]

    # 4) Junta con coma (o deja string vacío si no hay fechas)
    fecha_rad = ", ".join(fechas) if fechas else ""
    # Comisión
    com = extract_section(txt, "COMISION", "PONENTE").title()

    # Ponente
    pon = extract_section(txt, "PONENTE", "PUBLICACION")
    if not pon:
        pon = extract_section(txt, "PONENTE", "ESTADO ACTUAL")
    pon = pon or "N/A"

    # Publicaciones Gaceta
    pub = extract_section(txt, "PUBLICACION", "ESTADO ACTUAL")
    pubs = re.findall(r"(\d{1,3}/\d{2,4})", pub)
    publica = ", ".join(pubs) if pubs else ""

    # Estado Actual
    est = extract_section(txt, "ESTADO ACTUAL", "")

    rows.append({
        'Tipo de Proyecto':    tipo,
        'Proyecto No':         proj_no,
        'Proyecto Año':        proj_year,
        'Camara No':           cam_no,
        'Camara Año':          cam_year,
        'Senado No':           sen_no,
        'Senado Año':          sen_year,
        'Titulo':              titulo,
        'Autor':               autor,
        'Fecha Rad.':          fecha_rad,
        'Comision':            com,
        'Ponente':             pon,
        'Publica Gaceta':      publica,
        'Estado Actual':       est
    })

# 3) DataFrame y exportar
df = pd.DataFrame(rows)
for col in df.select_dtypes(include='object'):
    df[col] = df[col].apply(lambda s: ILLEGAL_CHARACTERS_RE.sub('', s) if isinstance(s,str) else s)
df.to_excel('fichas_1993_mejorado.xlsx', index=False)


In [1]:
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("Versión CUDA:", torch.version.cuda)
print("Nombre de GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No hay GPU")


CUDA disponible: True
Versión CUDA: 11.8
Nombre de GPU: NVIDIA GeForce RTX 4060 Laptop GPU
